# Train Residual Soft DQN on Colab

Clones a selected Git ref of `RQL-Comparison`, creates an isolated virtualenv, and runs residual customization via `train.py`:

```bash
python -W ignore train.py --algo dqn_soft_residual --env ENV --prior-model-path PRIOR -f LOG_FOLDER
```

**Defaults (merge courtesy residual):**
- Env: `merge-ME-basic-AddCourtesyReward-v0` (add-on only: `-sqrt` envelope penetration)
- Prior: your trained `merge-ME-basic` DQN_ME zip

For highway right-lane residual, set:
- `ENV_NAME = highway-ME-basic-AddRightReward-v0`
- `PRIOR_MODEL_PATH` to a highway ME-basic prior zip

**Logs / checkpoints** go under `LOG_FOLDER/{algo}/{env}_{run}/` on Drive (`-f`). Set `RESUME=True` to continue from the latest `rl_model_*_steps.zip` or `best_model.zip` under that algo+env prefix.

Prefer a GPU runtime. `DEVICE="auto"` passes `--device cuda` when `torch.cuda.is_available()` in the training venv. Push a ref that includes `MergeEnvMEAddCourtesyReward` (merge) or the highway AddRight env before running.

In [ ]:
# Mount Google Drive. Colab will prompt you to authorize access.
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Training parameters (edit this cell before running).
REPO_URL = "https://github.com/thowell332/RQL-Comparison.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}

ALGO = "dqn_soft_residual"  #@param {type:"string"}
ENV_NAME = "merge-ME-basic-AddCourtesyReward-v0"  #@param {type:"string"}
# Absolute Drive path or path relative to the cloned repo.
PRIOR_MODEL_PATH = "/content/drive/MyDrive/aaai2027/RQL-Comparison/models/merge_basic_final_seed1/policies/final/model.zip"  #@param {type:"string"}

N_TIMESTEPS = 500000  #@param {type:"integer"}
SAVE_FREQ = 10000  #@param {type:"integer"}
EVAL_FREQ = 25000  #@param {type:"integer"}
SEED = 1  #@param {type:"integer"}
# auto → cuda if available else cpu. Force "cuda" or "cpu" if needed.
DEVICE = "auto"  #@param {type:"string"}

# Root for rl_zoo3 logs: {LOG_FOLDER}/{ALGO}/{ENV_NAME}_{run}/
LOG_FOLDER = "/content/drive/MyDrive/aaai2027/RQL-Comparison/logs"  #@param {type:"string"}
RESUME = True  #@param {type:"boolean"}

In [ ]:
# Clone the requested ref and create a Colab-compatible training environment.
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/RQL-Comparison")
VENV_DIR = Path("/content/venvs/rql-comparison-residual")

os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OFFSCREEN_RENDERING"] = "1"

for path in (PROJECT_DIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_REF], check=True)


def create_venv(venv_dir: Path) -> Path:
    """Create a Colab-friendly venv that reuses system site packages (incl. CUDA torch)."""
    venv_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["apt-get", "install", "-y", "python3-venv", f"python{sys.version_info.major}.{sys.version_info.minor}-venv"],
        check=False,
    )
    result = subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", str(venv_dir)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("stdlib venv failed; falling back to virtualenv")
        print(result.stderr)
        if venv_dir.exists():
            shutil.rmtree(venv_dir)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
        subprocess.run(
            [sys.executable, "-m", "virtualenv", "--system-site-packages", str(venv_dir)],
            check=True,
        )
    return venv_dir / "bin" / "python"


VENV_PYTHON = create_venv(VENV_DIR)
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)

# Local stable_baselines3 / highway_env / rl_zoo3 come from PROJECT_DIR via cwd/PYTHONPATH.
# gym==0.26.2 matches merge prior checkpoints (RandomNumberGenerator pickles).
TRAIN_PACKAGES = [
    "numpy<2", "gym==0.26.2", "scipy", "pandas", "cloudpickle",
    "matplotlib", "tensorboard", "tensorboardX", "pygame", "tqdm",
    "rich", "pyyaml", "optuna",
    # Required by rl_zoo3.utils (imported from train.py).
    "huggingface_sb3>=2.2.1,<3",
]
result = subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "install", *TRAIN_PACKAGES],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise subprocess.CalledProcessError(result.returncode, result.args)

entry_point = PROJECT_DIR / "train.py"
merge_env = PROJECT_DIR / "highway_env" / "envs" / "merge_env.py"
residual_yml = PROJECT_DIR / "hyperparams" / "dqn_soft_residual.yml"
if not entry_point.is_file():
    raise FileNotFoundError(f"Selected Git ref does not contain {entry_point}")
if ENV_NAME.startswith("merge") and "MergeEnvMEAddCourtesyReward" not in merge_env.read_text():
    raise RuntimeError(
        f"{merge_env} is missing MergeEnvMEAddCourtesyReward. "
        f"Checkout a ref that includes the merge courtesy residual env."
    )
if ENV_NAME not in residual_yml.read_text() and ALGO == "dqn_soft_residual":
    print(
        f"Warning: {ENV_NAME} not found in {residual_yml.name}; "
        f"rl_zoo3 may fall back to another hyperparam block or fail."
    )

print(f"Project: {PROJECT_DIR}")
print(f"Python:  {VENV_PYTHON}")
print(f"Ref:     {REPO_REF}")
# Report torch CUDA visibility inside the training venv (Colab T4 should be True).
cuda_probe = subprocess.run(
    [
        str(VENV_PYTHON), "-c",
        "import torch; print(torch.__version__); print(torch.cuda.is_available()); "
        "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a')",
    ],
    capture_output=True,
    text=True,
    check=False,
)
print("Torch/CUDA probe:")
print(cuda_probe.stdout or cuda_probe.stderr)
if cuda_probe.returncode != 0:
    print("Warning: could not probe torch CUDA in the training venv.")


In [ ]:
# Resolve prior path, optional resume checkpoint, run train.py with logs on Drive.
import json
import os
import subprocess
from pathlib import Path


def resolve_path(value: str) -> Path:
    path = Path(value).expanduser()
    return path if path.is_absolute() else PROJECT_DIR / path


def resolve_prior_zip(path: Path) -> Path:
    if path.is_file() and path.suffix == ".zip":
        return path
    if path.with_suffix(".zip").is_file():
        return path.with_suffix(".zip")
    if path.is_dir():
        for name in ("model.zip", "best_model.zip"):
            candidate = path / name
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(
        f"Prior model not found at {path} (expected a .zip or a directory with model.zip)."
    )


def find_latest_residual_checkpoint(log_folder: Path, algo: str, env_name: str) -> Path | None:
    """Latest rl_model_*_steps.zip or best_model.zip under log_folder/algo/env_name_*."""
    algo_dir = log_folder / algo
    if not algo_dir.is_dir():
        return None
    run_dirs = sorted(
        [p for p in algo_dir.iterdir() if p.is_dir() and p.name.startswith(f"{env_name}_")],
        key=lambda p: p.stat().st_mtime,
    )
    if not run_dirs:
        return None
    run_dir = run_dirs[-1]
    step_ckpts = sorted(run_dir.glob("rl_model_*_steps.zip"), key=lambda p: p.stat().st_mtime)
    if step_ckpts:
        return step_ckpts[-1]
    best = run_dir / "best_model.zip"
    if best.is_file():
        return best
    # Final save named after the env id (no .zip suffix in some runs).
    for candidate in (run_dir / f"{env_name}.zip", run_dir / env_name):
        if candidate.is_file():
            return candidate
    return None


def resolve_train_device(requested: str) -> str:
    """Pick cuda when available; honor explicit DEVICE=cuda/cpu."""
    requested = (requested or "auto").strip().lower()
    probe = subprocess.run(
        [str(VENV_PYTHON), "-c", "import torch; print('1' if torch.cuda.is_available() else '0')"],
        capture_output=True,
        text=True,
        check=False,
    )
    cuda_ok = probe.returncode == 0 and probe.stdout.strip().endswith("1")
    if requested == "auto":
        device = "cuda" if cuda_ok else "cpu"
    else:
        device = requested
    if device.startswith("cuda") and not cuda_ok:
        raise RuntimeError(
            "DEVICE requests CUDA but torch.cuda.is_available() is False in the training venv. "
            "In Colab: Runtime → Change runtime type → GPU (T4), then re-run the setup cell. "
            f"Probe output: {probe.stdout!r} {probe.stderr!r}"
        )
    return device


train_device = resolve_train_device(DEVICE)
print(f"Training device: {train_device} (DEVICE={DEVICE!r})")

prior_path = resolve_prior_zip(resolve_path(PRIOR_MODEL_PATH))
log_folder = Path(LOG_FOLDER).expanduser()
log_folder.mkdir(parents=True, exist_ok=True)

trained_agent = None
if RESUME:
    trained_agent = find_latest_residual_checkpoint(log_folder, ALGO, ENV_NAME)
    if trained_agent is not None:
        print(f"Resuming from {trained_agent}")
    else:
        print(f"No residual checkpoint under {log_folder / ALGO}; starting a new run.")

manifest = {
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "algo": ALGO,
    "env_name": ENV_NAME,
    "prior_model_path": str(prior_path),
    "n_timesteps": N_TIMESTEPS,
    "save_freq": SAVE_FREQ,
    "eval_freq": EVAL_FREQ,
    "seed": SEED,
    "device": train_device,
    "log_folder": str(log_folder),
    "resume": RESUME,
    "trained_agent": str(trained_agent) if trained_agent else None,
}
run_meta_dir = log_folder / ALGO
run_meta_dir.mkdir(parents=True, exist_ok=True)
(run_meta_dir / f"{ENV_NAME}_last_launch.json").write_text(json.dumps(manifest, indent=2))

command = [
    str(VENV_PYTHON), "-W", "ignore", str(PROJECT_DIR / "train.py"),
    "--algo", ALGO,
    "--env", ENV_NAME,
    "--prior-model-path", str(prior_path),
    "-f", str(log_folder),
    "-n", str(N_TIMESTEPS),
    "--save-freq", str(SAVE_FREQ),
    "--eval-freq", str(EVAL_FREQ),
    "--seed", str(SEED),
    "--device", train_device,
]
if trained_agent is not None:
    command.extend(["-i", str(trained_agent)])

print("$", " ".join(command))
run_env = {
    **os.environ,
    "PYTHONUNBUFFERED": "1",
    "PYTHONPATH": str(PROJECT_DIR),
}
log_path = run_meta_dir / f"{ENV_NAME}_train.log"
log_mode = "a" if RESUME and log_path.exists() else "w"
with log_path.open(log_mode, buffering=1) as log:
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=run_env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

print(f"Logs under: {log_folder / ALGO}")

In [ ]:
# List residual artifacts under the Drive log folder for this algo/env.
algo_dir = Path(LOG_FOLDER).expanduser() / ALGO
if not algo_dir.is_dir():
    print(f"No log dir yet: {algo_dir}")
else:
    for artifact in sorted(p for p in algo_dir.rglob("*") if p.is_file() and ENV_NAME in str(p)):
        print(f"{artifact.relative_to(algo_dir)}  ({artifact.stat().st_size:,} bytes)")